# 🤖 Multimodal Chatbot: a code walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jameel-learner/ai_training/blob/main/sprint1/day04/multimodal_chatbot_walkthrough.ipynb)

This notebook takes apart the code in **`multimodal_chatbot.py`**, one piece at a time.
**You don't need to know how to code.** You'll press ▶️, look at the result, and read what happened.

### How to use this notebook
- Each grey box is a **code cell**. Click it, then press the **▶️ button** on its left (or **Shift + Enter**).
- **Run the cells from top to bottom.** Later cells use things that earlier cells set up.
- Lines starting with **`#`** are **comments**: notes written for people. The computer ignores them.
- A cell is finished when its ▶️ stops spinning and the result appears below it.

### What we'll build
The chatbot is a **team of specialist AI models**, each good at one conversion:

| # | Specialist | Converts | Function in `multimodal_chatbot.py` |
|---|---|---|---|
| 1 | 💬 Conversationalist | Text → Text | `chat_reply()` |
| 2 | 🏷️ Labeller | Image → Text (caption) | `caption_image()` |
| 3 | 👁️ Observer | Image + Question → Answer | `vision_reply()` |
| 4 | 🗣️ Narrator | Text → Speech | `speak()` |
| 5 | 👂 Listener | Speech → Text | `transcribe()` |
| 6 | 🧑‍✈️ Coordinator | Sends each message to the right specialist | the main part of the app |
| 7 | 🎨 Artist | Text → Image | `generate_image()` |

> 💡 **Optional speed boost:** in the menu choose **Runtime → Change runtime type → T4 GPU → Save**.
> Everything works without it, but the Artist (section 7) goes from about a minute to a few seconds.

## Step 0: Set up the toolbox

We install a few free libraries. A **library** is a ready-made toolbox of code someone else wrote:
- **transformers** (from Hugging Face): loads and runs AI models in one line
- **diffusers** (from Hugging Face): runs image-generating models
- **soundfile**: reads and writes audio

⏳ This takes about a minute. If Colab shows a **"Restart session"** button afterwards, click it
and then carry on with the next cell.

In [ ]:
%pip install -q "transformers>=4.45,<5" "huggingface_hub>=0.34,<1.0" "diffusers>=0.30,<0.36" accelerate soundfile matplotlib

In [ ]:
import io
import numpy as np                       # maths with long lists of numbers
import requests                          # downloads things from the internet
import torch                             # the engine that runs the AI models
import matplotlib.pyplot as plt          # draws charts
from PIL import Image                    # opens and shows pictures
from IPython.display import Audio, display
from transformers import pipeline        # ⭐ the one-line "load a model for a task" tool

# Is there a GPU (graphics chip) available? It makes AI much faster.
DEVICE = 0 if torch.cuda.is_available() else -1   # 0 = first GPU, -1 = CPU
print("Running on:", "GPU 🚀" if DEVICE == 0 else "CPU 🐢 (works fine, just slower)")

> 🔑 **Hugging Face token (optional):** all the models here are free and public, so no account is needed.
> If downloads are slow, click the **🔑 key icon** in Colab's left sidebar, add a secret named
> **`HF_TOKEN`** with your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens),
> and switch on **Notebook access**. In the app, the same token goes in `.env` or Streamlit's secrets.

## 1. 💬 The Conversationalist: text → text
*In the app: `chat_reply()`*

**`pipeline(...)`** loads a model ready for a specific **task** in one line. Here the task is
`"text-generation"` and the model is **Qwen 2.5**, a small chat model with **0.5 billion parameters**.
(Parameters are roughly the size of the model's "brain". ChatGPT-class models have hundreds of billions.)

⏳ The first run downloads the model (~1 GB). After that it loads from storage.

In [ ]:
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=DEVICE)

A conversation is just a **list of messages**. Each message has a **role**:
- `system`: instructions about how the bot should behave (its "job description")
- `user`: you
- `assistant`: the bot

In [ ]:
conversation = [
    {"role": "system", "content": "You are a helpful, concise assistant. Keep answers short."},
    {"role": "user",   "content": "What is the capital of India?"},   # ✏️ try your own question
]

result = chat(conversation, max_new_tokens=100)        # max_new_tokens = longest answer allowed
answer = result[0]["generated_text"][-1]["content"]    # the last message is the bot's new reply
print("🤖", answer)

### How does the bot "remember" the conversation?
**It doesn't.** The model has no memory at all. Instead, the app **sends the whole conversation again**
with every new message. In the app, `history_for_llm()` builds that list.

Watch: the next question says *"it"*, which only makes sense if the bot sees the earlier messages.

In [ ]:
conversation.append({"role": "assistant", "content": answer})              # add the bot's answer
conversation.append({"role": "user", "content": "What is it famous for?"}) # "it" = the city from before

answer = chat(conversation, max_new_tokens=100)[0]["generated_text"][-1]["content"]
print("🤖", answer)

> ✏️ **Try it:** change the question in the first chat cell (keep the quote marks) and run both cells again.
> In the app, answers appear **word by word** ("streaming"). It's the same model, just shown as it writes.

## 2. 🏷️ The Labeller: image → text
*In the app: `caption_image()`*

First we need a picture. This one is a well-known sample photo used in AI tutorials.

In [ ]:
IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"   # ✏️ any image link works
photo = Image.open(requests.get(IMAGE_URL, stream=True).raw).convert("RGB")
photo.thumbnail((448, 448))   # shrink it: smaller pictures are faster for the models
display(photo)

In [ ]:
USE_MY_OWN_PHOTO = False   # ✏️ change to True and run this cell to upload a photo from your computer

if USE_MY_OWN_PHOTO:
    from google.colab import files
    uploaded = files.upload()
    photo = Image.open(io.BytesIO(next(iter(uploaded.values())))).convert("RGB")
    photo.thumbnail((448, 448))
    display(photo)

Now the Labeller. Same one-line `pipeline(...)` pattern, but a different **task** (`"image-to-text"`)
and a different **model** (**BLIP**, trained on millions of images paired with captions).

In [ ]:
captioner = pipeline("image-to-text", model="Salesforce/blip-image-captioning-base", device=DEVICE)

caption = captioner(photo)[0]["generated_text"]
print("🏷️ Caption:", caption)

The Labeller can only **describe** a picture. It can't answer questions about it. That's the Observer's job.

## 3. 👁️ The Observer: image + question → answer
*In the app: `vision_reply()`*

This is a **vision-language model** (SmolVLM). It understands pictures and words **together**.

In [ ]:
observer = pipeline("image-text-to-text", model="HuggingFaceTB/SmolVLM-256M-Instruct", device=DEVICE)

# ⚡ Speed trick used in the app: by default this model cuts every picture into many tiles,
# which is ~15x slower on a normal computer. We tell it to look at the picture once, at a modest size.
observer.processor.image_processor.do_image_splitting = False
observer.processor.image_processor.size = {"longest_edge": 512}

⭐ **This is the heart of multimodal AI:** a single message that contains **both a picture and text**.

In [ ]:
question = "How many cats are in this picture, and what are they doing?"   # ✏️ ask anything

message = [{"role": "user", "content": [
    {"type": "image", "image": photo},      # 👁️ what it sees
    {"type": "text",  "text": question},    # 💬 what we ask
]}]

answer = observer(text=message, max_new_tokens=80, return_full_text=False)[0]["generated_text"]
print("👁️", answer.strip())

## 4. 🗣️ The Narrator: text → speech
*In the app: `speak()`*

In [ ]:
narrator = pipeline("text-to-speech", model="facebook/mms-tts-eng", device=DEVICE)

speech = narrator("Hello! I am the narrator of this multimodal chatbot.")   # ✏️ change the words
display(Audio(speech["audio"].squeeze(), rate=speech["sampling_rate"]))     # ▶️ press play

### What does sound look like to a computer?
Just a **very long list of numbers**: how far the speaker moves in and out, thousands of times a second.
That speed is the **sampling rate**.

In [ ]:
wave = speech["audio"].squeeze()
print(f"{len(wave):,} numbers, played at {speech['sampling_rate']:,} numbers per second "
      f"= {len(wave) / speech['sampling_rate']:.1f} seconds of speech")

plt.figure(figsize=(10, 2))
plt.plot(wave, linewidth=0.5)
plt.title("The sound wave the Narrator just made")
plt.axis("off")
plt.show()

## 5. 👂 The Listener: speech → text
*In the app: `transcribe()`*

**Whisper** (made by OpenAI, free on Hugging Face) turns speech into text. It expects sound at exactly
**16,000 numbers per second**, so, just like the app, we convert the audio first.

To test it without a microphone, the Listener will hear **what the Narrator just said**.

In [ ]:
listener = pipeline("automatic-speech-recognition", model="openai/whisper-tiny", device=DEVICE)

def to_16khz(wave, rate):
    """Convert a sound wave to 16,000 numbers per second (the same step as in transcribe())."""
    wave = np.asarray(wave, dtype=np.float32).squeeze()
    new_length = int(len(wave) * 16000 / rate)
    return np.interp(np.linspace(0, len(wave) - 1, new_length), np.arange(len(wave)), wave).astype(np.float32)

heard = listener({"raw": to_16khz(speech["audio"], speech["sampling_rate"]), "sampling_rate": 16000})
print("👂 Heard:", heard["text"])

In [ ]:
USE_MY_OWN_RECORDING = False   # ✏️ True, then upload a short voice recording (.wav, .mp3 or .m4a)

if USE_MY_OWN_RECORDING:
    from google.colab import files
    uploaded = files.upload()
    print("👂 Heard:", listener(next(iter(uploaded)))["text"])

> 🤔 **Did it hear every word right?** This is *whisper-tiny*, the smallest version, so it sometimes mishears.
> Bigger versions (`whisper-small`, `whisper-large`) are more accurate but slower.

## 6. 🧑‍✈️ The Coordinator: putting the team together
*In the app: the main part of the script, below the specialist functions*

Every time you send a message, the app asks a few simple questions and passes the work to the right specialist:

1. **Is it a voice recording?** → the 👂 Listener turns it into text
2. **Is there a photo?** → the 👁️ Observer answers. **No photo?** → the 💬 Conversationalist answers
3. **Is "Speak replies" on?** → the 🗣️ Narrator reads the answer aloud

Here's a mini version of that logic:

In [ ]:
def multimodal_chatbot(text="", photo=None, voice=None, speak_reply=False):
    """A mini version of what the Streamlit app does with every message."""
    # 1. Voice in?  → the Listener turns it into text
    if voice is not None:
        text = listener(voice)["text"].strip()
        print("👂 You said:", text)

    # 2. Photo?  → the Observer.   No photo?  → the Conversationalist.
    if photo is not None:
        message = [{"role": "user", "content": [
            {"type": "image", "image": photo},
            {"type": "text",  "text": text or "Describe this image."},
        ]}]
        reply = observer(text=message, max_new_tokens=80, return_full_text=False)[0]["generated_text"].strip()
    else:
        conversation = [{"role": "system", "content": "You are a helpful, concise assistant."},
                        {"role": "user",   "content": text}]
        reply = chat(conversation, max_new_tokens=100)[0]["generated_text"][-1]["content"]
    print("🤖", reply)

    # 3. Speak the reply?  → the Narrator
    if speak_reply:
        spoken = narrator(reply)
        display(Audio(spoken["audio"].squeeze(), rate=spoken["sampling_rate"]))

Now let's send it three different kinds of message:

In [ ]:
print("━━ A typed question ━━")
multimodal_chatbot("Give me one fun fact about elephants.")

print("\n━━ A photo with a question ━━")
multimodal_chatbot("What colour is the blanket?", photo=photo)

In [ ]:
print("━━ A spoken question, with a spoken answer ━━")
# No microphone needed: the Narrator "asks" the question out loud, and the Listener hears it.
question_audio = narrator("What is the largest planet in our solar system?")
voice = {"raw": to_16khz(question_audio["audio"], question_audio["sampling_rate"]), "sampling_rate": 16000}

multimodal_chatbot(voice=voice, speak_reply=True)

> 🔁 Notice the chain in the last example: **speech → text → answer → speech**, made by
> three different models handing work to each other. **Text is the common language between them.**

## 7. 🎨 The Artist: text → image
*In the app: `generate_image()` (type `/imagine ...` in the chat)*

This model (**SD-Turbo**) uses a technique called **diffusion**:
1. It starts from a canvas of random noise, like TV static 📺
2. Step by step, it removes the noise, guided by your description, until a picture appears
3. Most models need 20–50 steps. *Turbo* was trained to do it in **just 1 step**

It's the biggest model in the team (a ~2.5 GB download), so first we unload the others to free up memory.

In [ ]:
import gc
del chat, captioner, observer, listener    # done with these: free the memory for the big one
gc.collect()
if DEVICE == 0:
    torch.cuda.empty_cache()
print("Memory freed ✅  (re-run the earlier sections if you want to use those specialists again)")

In [ ]:
from diffusers import AutoPipelineForText2Image

artist = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    variant="fp16",                                                  # the smaller download
    torch_dtype=torch.float16 if DEVICE == 0 else torch.float32,     # GPUs can use "half precision"
).to("cuda" if DEVICE == 0 else "cpu")

In [ ]:
prompt = "a cat astronaut on the moon, digital art"   # ✏️ describe anything you like

picture = artist(prompt, num_inference_steps=1, guidance_scale=0.0).images[0]
display(picture)

> ✏️ **Try it:** change the description and run the cell again. Each run starts from different random
> noise, so you'll get a new picture every time, even with the same words.

## 8. From this notebook to the real app

The specialist cells in this notebook use the same models and steps as the functions in `multimodal_chatbot.py`:

| Notebook section | Function in the app |
|---|---|
| 1. Conversationalist | `chat_reply()` + `history_for_llm()` (the memory list) |
| 2. Labeller | `caption_image()` |
| 3. Observer | `vision_reply()` |
| 4. Narrator | `speak()` |
| 5. Listener | `transcribe()` |
| 6. Coordinator | the main part of the script |
| 7. Artist | `generate_image()` |

**So what's the rest of the app?** It's the **screen**: chat bubbles, the 📎 button, the 🎙️ microphone and
the sidebar. It's built with **Streamlit**, which turns each screen element into about one line of Python:

```python
prompt = st.chat_input("Ask something or attach an image", accept_file=True)   # the message box + 📎
voice  = st.audio_input("🎙️ Ask by voice")                                     # the microphone
st.write_stream(tokens)                                                          # word-by-word typing
```

Two more tricks in the app you've now seen the reason for:
- **`@st.cache_resource`** loads each model **once** and reuses it. Without it, the app would reload the models
  every time you click something, because Streamlit re-runs the whole script on every click.
- **Cloud "lite" mode:** the free Streamlit Cloud server has only ~2.7 GB of memory. So the app switches off
  the Labeller and the Artist there, and uses the Observer for chat too.

**▶️ To run the real app**, see sections 7 and 8 of the `README.md` in the same folder.

## 🧠 Recap

1. **Multimodal** means handling more than one type of information: text, images and sound.
2. There's no single all-in-one AI here, just **a team of specialist models**, each one loaded with the same one-line `pipeline(...)` pattern.
3. Models have **no memory**. The app re-sends the conversation each time.
4. A **vision-language model** reads a message that mixes a picture and text.
5. Sound and pictures are just **numbers** to a computer, and **text** is the common language the specialists pass to each other.

### 🧪 Experiments to try
- Ask the Observer a tricky question about your own photo (section 2's upload cell)
- Make the Narrator say a tongue-twister, then see whether the Listener gets it right
- Swap `whisper-tiny` for `openai/whisper-base` in section 5. Is it more accurate? Slower?
- Change the Conversationalist's `system` message to *"You are a pirate"* 🏴‍☠️ and chat again